# Pathshala — Notebook 2: HSC books and final archive

This notebook restores the SSC artifacts from Notebook 1, OCRs and indexes `hsc-2026-bangla-1.pdf` and `hsc-2026-bangla-2.pdf`, then exports one final `pathshala-indexes.zip` containing all four books.

Attach the HSC PDFs (or the same private dataset used by Notebook 1), `ssc-indexes.zip`, and `ssc-ocr-checkpoints.zip` as Kaggle inputs. Also attach this repository as a Kaggle dataset or upload `kaggle/build_indexes.py`. Use a GPU accelerator and enable Internet for the first model download.

In [ ]:
!nvidia-smi


In [ ]:
!apt-get update -qq
!apt-get install -y -qq tesseract-ocr tesseract-ocr-ben


In [ ]:
!python -m pip install -q --upgrade-strategy only-if-needed pymupdf==1.26.4 'sentence-transformers>=5.1.1,<6.0.0' tqdm==4.67.1


In [ ]:
!python -m pip install -q virtualenv
!python -m virtualenv --clear --system-site-packages /kaggle/working/paddle-env
!/kaggle/working/paddle-env/bin/python -m pip install -q --upgrade pip setuptools wheel wrapt
!/kaggle/working/paddle-env/bin/python -m pip install -q paddlepaddle-gpu==3.3.0 -i https://www.paddlepaddle.org.cn/packages/stable/cu130/
!/kaggle/working/paddle-env/bin/python -m pip install -q 'paddleocr[doc-parser]>=3.6.0,<3.7.0' pymupdf==1.26.4 'numpy>=1.26,<3' tqdm==4.67.1


In [ ]:
import subprocess
import torch

print('Torch:', torch.__version__)
print('Torch CUDA available:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'Enable a Kaggle GPU before continuing'
paddle_check = (
    'import paddle; '
    'print("Paddle:", paddle.__version__); '
    'print("Paddle compiled with CUDA:", paddle.device.is_compiled_with_cuda()); '
    'print("Paddle device:", paddle.device.get_device()); '
    'assert paddle.device.is_compiled_with_cuda(); paddle.utils.run_check()'
)
subprocess.run(['/kaggle/working/paddle-env/bin/python', '-c', paddle_check], check=True)
print('Torch and Paddle runtimes are ready.')


In [ ]:
from pathlib import Path
import shutil

candidates = list(Path('/kaggle/input').rglob('build_indexes.py'))
print('Indexer candidates:')
for path in candidates:
    print(' -', path)
assert candidates, (
    'No build_indexes.py found. Attach the repository as a Kaggle dataset '
    'or upload kaggle/build_indexes.py.'
)
source = next((path for path in candidates if path.parent.name == 'kaggle'), candidates[0])
shutil.copy2(source, '/kaggle/working/build_indexes.py')
print('Copied:', source)


In [ ]:
from pathlib import Path

HSC_IDS = ['hsc-2026-bangla-1', 'hsc-2026-bangla-2']
pdf_paths = {}
for book_id in HSC_IDS:
    matches = list(Path('/kaggle/input').rglob(f'{book_id}.pdf'))
    print(book_id, matches)
    assert len(matches) == 1, f'Expected exactly one {book_id}.pdf'
    assert matches[0].stat().st_size <= 100 * 1024 * 1024, f'{book_id}.pdf is over 100 MB'
    pdf_paths[book_id] = matches[0]
print('HSC input PDFs are ready.')


## Restore Notebook 1 artifacts

The index ZIP preserves the SSC source PDFs, manifests, chunks, and vectors. The checkpoint ZIP lets this notebook verify the complete 1,186-page OCR set before adding HSC. Both ZIPs must be attached as Kaggle inputs.

In [ ]:
from pathlib import Path
import shutil

index_inputs = list(Path('/kaggle/input').rglob('ssc-indexes.zip'))
assert index_inputs, 'Attach the ssc-indexes.zip output from Notebook 1'
output_root = Path('/kaggle/working/pathshala-indexes')
output_root.mkdir(parents=True, exist_ok=True)
if not (output_root / 'indexes').is_dir():
    shutil.unpack_archive(index_inputs[0], output_root, 'zip')
print('Restored SSC index archive:', index_inputs[0])
assert (output_root / 'indexes' / 'ssc-2026-bangla-1' / 'active.json').is_file()
assert (output_root / 'indexes' / 'ssc-2026-bangla-2' / 'active.json').is_file()


In [ ]:
checkpoint_inputs = list(Path('/kaggle/input').rglob('ssc-ocr-checkpoints.zip'))
assert checkpoint_inputs, 'Attach the ssc-ocr-checkpoints.zip output from Notebook 1'
ocr_root = Path('/kaggle/working/pathshala-build/ocr')
ocr_root.mkdir(parents=True, exist_ok=True)
if not (ocr_root / 'ssc-2026-bangla-1').is_dir():
    shutil.unpack_archive(checkpoint_inputs[0], ocr_root, 'zip')
print('Restored SSC OCR checkpoints:', checkpoint_inputs[0])
assert (ocr_root / 'ssc-2026-bangla-1').is_dir()
assert (ocr_root / 'ssc-2026-bangla-2').is_dir()


## OCR the HSC group

The command is resumable. Rerun it if a session ends before all HSC pages receive checkpoints.

In [ ]:
!PADDLE_PDX_DISABLE_MODEL_SOURCE_CHECK=True /kaggle/working/paddle-env/bin/python /kaggle/working/build_indexes.py --stage ocr --book hsc --ocr-engine paddle-vl --ocr-workers 1 --dpi 250


In [ ]:
import json
import pymupdf

expected_pages = {'ssc-2026-bangla-1': 265, 'ssc-2026-bangla-2': 219}
review_pages = []
total = 0
for book_id, expected in expected_pages.items():
    checkpoints = sorted((ocr_root / book_id).glob('*.json'))
    print(book_id, 'checkpoints:', len(checkpoints), 'expected:', expected)
    assert len(checkpoints) == expected, f'{book_id} restored OCR is incomplete'
    total += len(checkpoints)
    for checkpoint in checkpoints:
        record = json.loads(checkpoint.read_text(encoding='utf-8'))
        if record.get('status') != 'ok':
            review_pages.append((book_id, record.get('page')))
for book_id in HSC_IDS:
    with pymupdf.open(pdf_paths[book_id]) as pdf:
        expected = len(pdf)
    checkpoints = sorted((ocr_root / book_id).glob('*.json'))
    print(book_id, 'checkpoints:', len(checkpoints), 'expected:', expected)
    assert len(checkpoints) == expected, f'{book_id} OCR is incomplete'
    total += len(checkpoints)
    for checkpoint in checkpoints:
        record = json.loads(checkpoint.read_text(encoding='utf-8'))
        if record.get('status') != 'ok':
            review_pages.append((book_id, record.get('page')))
print('Total OCR checkpoints:', total, 'expected:', 1186)
print('Automatically flagged pages:', review_pages)
assert total == 1186, 'The four books should contain 1,186 pages'


## Add HSC vectors and package all four books

Because the SSC index tree was restored before this command, the indexer keeps the SSC manifests and adds the two HSC manifests to the final archive.

In [ ]:
!python /kaggle/working/build_indexes.py --stage index --book hsc --ocr-engine paddle-vl --ocr-workers 1 --embed-batch-size 16 --dpi 250


In [ ]:
from IPython.display import FileLink, display
from pathlib import Path
import json
import zipfile

ALL_IDS = [
    'ssc-2026-bangla-1',
    'ssc-2026-bangla-2',
    'hsc-2026-bangla-1',
    'hsc-2026-bangla-2',
]
archive = Path('/kaggle/working/pathshala-indexes.zip')
assert archive.is_file() and zipfile.is_zipfile(archive), 'Final index ZIP was not created'
with zipfile.ZipFile(archive) as bundle:
    assert bundle.testzip() is None, 'The final ZIP is corrupt'
    names = bundle.namelist()
    for book_id in ALL_IDS:
        assert any(f'indexes/{book_id}/active.json' in name for name in names), f'Missing {book_id}'
    summary = json.loads(bundle.read('build-summary.json'))
    found = {item['book_id'] for item in summary['indexes']}
    assert found == set(ALL_IDS), f'Unexpected summary books: {found}'
    print('ZIP entries:', len(names))
print(json.dumps(summary, indent=2, ensure_ascii=False))
print(f'Final ZIP size: {archive.stat().st_size / 1024**2:.1f} MB')
display(FileLink(str(archive)))
